In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities

## Widgets


In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source", "products","Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")


In [0]:
base_path = f"s3://sportsbar-dp-child-company-prac/{data_source}/*.csv"


## Reading from S3


In [0]:
df = (
    spark.read.format('csv')
    .option("header", True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)

In [0]:
display(df)

## Write data to Bronze Layer

In [0]:
df.write.format("delta")\
.option("delta.enableChangeDataFeed", True)\
.mode("overwrite")\
.saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

#### Reading data from bronze layer


In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)

In [0]:
    # Checking to see if there are any null values in the dataset
display(
    df.filter(
        sum(F.col(c).isNull().cast('int') for c in df.columns) > 0
    )
)

#### Transformations


In [0]:
# Checking duplicates for product_id
display(
    df.groupBy("product_id").count().filter(F.col("count") > 1)
)

In [0]:
# Checking records where there are duplicate product_id
duplicate_product_id = ['25891101','25891102']
display(
    df.filter(F.col("product_id").isin(duplicate_product_id)).orderBy(F.col("product_id"))
)

In [0]:
# Removing duplicates
df_bronze = df.dropDuplicates(['product_id'])

In [0]:
display(
    df_bronze.orderBy("product_id")
)

In [0]:
display(
    df_bronze.select("category").distinct()
)

#### Capitalizing Category to match Category of main table in Gold Layer

In [0]:

df_bronze = df_bronze.withColumn("category", 
F.when(F.col("category").isNull(), None)
.otherwise(F.initcap(F.col("category"))))

In [0]:
display(df_bronze.select("category").distinct())

In [0]:
#Fixing Protien -> Protein for "Category" column
fixed_map = {
    "Protien Bars":"Protein Bars"
}

In [0]:
df_bronze = df_bronze.replace(fixed_map, subset='category')


In [0]:
#Fixing Protien -> Protein in "product_name" column (REGEX)
df_bronze = df_bronze.withColumn(
    "product_name",
    F.regexp_replace(F.col("product_name"),"(?i)Protien", "Protein")
)


In [0]:
display(df_bronze)

#### Creating 'division' column to match main gold table


In [0]:
display(df_bronze.select("category").distinct())


In [0]:
df_bronze = df_bronze.withColumn(
    "division",
    F.when(F.col("category") == "Protein Bars",     "Nutrition Bars")
    .when(F.col("category") == "Recovery Dairy",    "Dairy & Recovery")
    .when(F.col("category") == "Energy Bars",       "Nutrition Bars")
    .when(F.col("category") == "Electrolyte Mix",   "Hydration & Electrolytes")
    .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")
    .when(F.col("category") == "Healthy Snacks",    "Healthy Snacks")
    .otherwise("Other")
)

In [0]:
display(df_bronze)

#### Product variant column, from product_name column

In [0]:
df_bronze = df_bronze.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\(([^)]+)\)", 1)
)

In [0]:
display(df_bronze)

#### Creating a unique identifier "product_code" 

In [0]:
df_bronze = (
    df_bronze.withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    .withColumn(
        "product_id",
        F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"),
                F.col("product_id").cast("string"))
        .otherwise(F.lit(999999).cast("string"))
    )
)

In [0]:
#Rename column
df_bronze = df_bronze.withColumnRenamed("product_name","product")

In [0]:
# Reordering columns
df_bronze = df_bronze.select(['product_code','division','category','product','variant','product_id','read_timestamp','file_name','file_size'])

In [0]:
display(df_bronze)

In [0]:
df_bronze.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')